# Train a multi-speaker Assamese Piper voice

Trains Piper's VITS model on [`kalita11/Assamese-TTS-corpus`](https://huggingface.co/datasets/kalita11/Assamese-TTS-corpus) — 76.8 hours, 5 speakers, CC-BY-4.0 — fine-tuned from the Bengali `bn_BD-google-medium` checkpoint (closest related language/script Piper already has).

**How to use this notebook:**
1. Runtime > Change runtime type > GPU (free tier: T4).
2. Run cells top to bottom.
3. Colab's free tier disconnects after ~90min idle or ~12hr total. When it does: just reopen this notebook and re-run all cells — the "resume" cell auto-detects your last checkpoint in Drive and continues from there. Nothing is lost.
4. Training will take multiple sessions across a day or more to reach good quality. Check progress periodically with the "listen to a sample" cell near the end.
5. Once happy with quality, run the export cell to produce the `.onnx` + `.onnx.json` files, download them, and drop them in `piper-tts-app/voices/`.

Everything persists in your Google Drive under `/content/drive/MyDrive/assamese-piper/` so you never lose progress between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/assamese-piper'
CACHE_DIR = f'{PROJECT_DIR}/cache'
AUDIO_DIR = f'{PROJECT_DIR}/wavs'
CKPT_DIR = f'{PROJECT_DIR}/checkpoints'
CONFIG_PATH = f'{PROJECT_DIR}/config.json'
CSV_PATH = f'{PROJECT_DIR}/metadata.csv'
BASE_CKPT_PATH = f'{PROJECT_DIR}/bn_BD-google-medium-base.ckpt'
for d in (PROJECT_DIR, CACHE_DIR, AUDIO_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print('Project dir ready:', PROJECT_DIR)

## 1. Install Piper's training package (first session only — fast on later resumes since pip caches)

In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null
![ -d /content/piper1-gpl ] || git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!pip install -q -e '.[train]'
!bash build_monotonic_align.sh
!python3 setup.py build_ext --inplace -q
print('Piper training package installed.')

## 2. Download the dataset and the Bengali base checkpoint (first session only — cached in Drive after)

In [ ]:
import os
if not os.path.exists(BASE_CKPT_PATH):
    print('Downloading Bengali base checkpoint (~900MB)...')
    !curl -sL -o "{BASE_CKPT_PATH}" "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/bn/bn_BD/google/medium/last.ckpt"
else:
    print('Base checkpoint already downloaded.')

In [ ]:
# Build metadata.csv + wavs/ from the HF dataset's parquet (audio bytes -> individual .wav files).
# Only runs the parts that are missing, so re-running after a disconnect is cheap.
import os, csv, io
from datasets import load_dataset
import soundfile as sf

if os.path.exists(CSV_PATH) and len(os.listdir(AUDIO_DIR)) > 1000:
    print('metadata.csv and wavs/ already built, skipping. Delete them in Drive to rebuild.')
else:
    print('Loading dataset (this streams ~77 hours of audio, may take a while the first time)...')
    ds = load_dataset('kalita11/Assamese-TTS-corpus', split='train')
    print('Rows:', len(ds))
    print('Columns:', ds.column_names)

    rows = []
    for i, ex in enumerate(ds):
        wav_name = f'utt{i:06d}.wav'
        wav_path = os.path.join(AUDIO_DIR, wav_name)
        if not os.path.exists(wav_path):
            audio = ex['audio']  # expects {'array':..., 'sampling_rate':...} from HF Audio feature
            sf.write(wav_path, audio['array'], audio['sampling_rate'])
        speaker = str(ex.get('speaker_id', ex.get('speaker', 'spk0')))
        text = ex['text'] if 'text' in ex else ex.get('transcript', '')
        rows.append((wav_name, speaker, text))
        if i % 2000 == 0:
            print(f'{i}/{len(ds)} processed')

    with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f, delimiter='|')
        for wav_name, speaker, text in rows:
            w.writerow([wav_name, speaker, text])
    print('Wrote', len(rows), 'rows to', CSV_PATH)

# IMPORTANT: inspect ds.column_names the first time this runs (printed above) and fix the
# 'text'/'speaker_id' field names in this cell if they don't match the actual dataset schema,
# then re-run. The dataset card didn't give exact column names, only that they exist.

## 3. Train (resumable — run this cell every session, including the first)

In [ ]:
import glob, os
%cd /content/piper1-gpl

# Resume from the most recent checkpoint saved in Drive, if any; otherwise start from the
# Bengali base checkpoint (fine-tuning, which the Piper docs say is much faster than scratch,
# even across languages).
existing = sorted(glob.glob(f'{CKPT_DIR}/*.ckpt'), key=os.path.getmtime)
resume_ckpt = existing[-1] if existing else BASE_CKPT_PATH
print('Resuming from:', resume_ckpt)

!python3 -m piper.train fit \
  --data.voice_name "as_IN-corpus-medium" \
  --data.csv_path "{CSV_PATH}" \
  --data.audio_dir "{AUDIO_DIR}" \
  --model.sample_rate 24000 \
  --data.espeak_voice "as" \
  --data.cache_dir "{CACHE_DIR}" \
  --data.config_path "{CONFIG_PATH}" \
  --data.batch_size 16 \
  --ckpt_path "{resume_ckpt}" \
  --trainer.default_root_dir "{CKPT_DIR}" \
  --trainer.max_epochs 2000

# batch_size 16 to be safe on a free-tier T4's ~15GB VRAM; raise to 32 if you get a bigger GPU.
# This will run until Colab disconnects you or you manually stop it (Runtime > Interrupt) —
# either way, just re-run this cell next session to continue from the latest checkpoint.

## 4. Listen to a sample from the current checkpoint (run any time to check progress)

In [ ]:
import glob, os, subprocess
from IPython.display import Audio, display

latest = sorted(glob.glob(f'{CKPT_DIR}/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
print('Using checkpoint:', latest)

test_onnx = '/content/preview.onnx'
!python3 -m piper.train.export_onnx --checkpoint "{latest}" --output-file "{test_onnx}"
!cp "{CONFIG_PATH}" "{test_onnx}.json"

!pip install -q piper-tts
from piper import PiperVoice
import wave, io
voice = PiperVoice.load(test_onnx)
buf = io.BytesIO()
with wave.open(buf, 'wb') as wf:
    voice.synthesize_wav('\u09a8\u09ae\u09b8\u09cd\u0995\u09be\u09b0, \u098f\u0987\u099f\u09cb \u098f\u0995 \u09aa\u09b0\u09c0\u0995\u09cd\u09b7\u09be\u0964', wf)  # placeholder Bengali-script test line - replace with a real Assamese sentence
buf.seek(0)
display(Audio(buf.read(), rate=voice.config.sample_rate))

## 5. Final export (once you're happy with the quality)

In [ ]:
import glob, os, shutil
latest = sorted(glob.glob(f'{CKPT_DIR}/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
FINAL_NAME = 'as_IN-corpus-medium'
final_onnx = f'/content/{FINAL_NAME}.onnx'
final_json = f'/content/{FINAL_NAME}.onnx.json'

!python3 -m piper.train.export_onnx --checkpoint "{latest}" --output-file "{final_onnx}"
shutil.copy(CONFIG_PATH, final_json)

shutil.copy(final_onnx, f'{PROJECT_DIR}/{FINAL_NAME}.onnx')
shutil.copy(final_json, f'{PROJECT_DIR}/{FINAL_NAME}.onnx.json')
print('Exported to Drive:', PROJECT_DIR)
print('Download these two files and put them in piper-tts-app/voices/:')
print(' -', final_onnx)
print(' -', final_json)
print()
print('This model has multiple speakers (one per unique speaker_id in the dataset).')
print('Check the config.json "speaker_id_map" for the exact names/ids to use per speaker.')